# Microsoft Agent Framework — Azure OpenAI (பதில் API)

இந்த குறியீட்டு உதாரணத்தில், நீங்கள் **Microsoft Agent Framework (MAF)** பயன்படுத்தி **Responses API** மூலம் **Azure OpenAI** ஆதரவு கொண்ட ஒரு எளிய முகவரியை உருவாக்குவீர்கள்.

> **மாற்றம் குறிப்பு:** இந்த உதாரணம் முந்தையதாக Semantic Kernel மற்றும் GitHub மாதிரிகளை பயன்படுத்தியது. தற்போது இது Microsoft Agent Frameworkக்கு மாறி, GitHub மாதிரிகள் (முடிவடையவுள்ளவைகள், ஜூலை 2026ல் ஓய்வுபெறுகின்றன) Azure OpenAI மூலம் மாற்றப்பட்டுள்ளன, இது Responses API க்கு ஆதரவாகும். MAF இல் உள்ள `OpenAIChatClient` Azure OpenAI இன் நிலையான `/openai/v1/` இடைமுகத்தைக் குறிக்கிறது மற்றும் Responses API ஐ இயல்பாகப் பயன்படுத்துகிறது.

இந்த உதாரணத்தில் நோக்கம், பிற குறியீட்டு உதாரணங்களில் பலவித முகவர் வடிவமைப்புகளை செயல்படுத்தும் போது பயன்படுத்தப்படும் படிகளைக் காட்டுவதாகும்.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## தேவையான Python தொகுதிகளை இறக்குமதி செய்க


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ஒரு கருவியை வரையறு

Microsoft Agent Framework-இல், ஒரு **கருவி** என்பது `@tool` என்ற அலங்காரம் செய்யப்பட்ட ஒரு சாதாரண Python செயல்பாடு ஆகும், இது முகவர் அழைக்க முடியும். கீழே, ஒரு சாதாரண விடுமுறை இருப்பிடம் திரும்ப அபரிமிதத்தைத் தவிர்க்கும் ஒரு கருவியை வரையறுக்கின்றோம்.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## முகவரியை உருவாக்குதல்

இங்கே, `TravelAgent` என்ற முகவரியை உருவாக்குகிறோம்.

இந்த உதாரணத்தில், நாம் மிகவும் அடிப்படையான வழிமுறைகளை பயன்படுத்துகிறோம். முகவரியின் நடத்தையில் என்ன மாற்றம் வருகிறது என்பதை கவனிக்க இந்த வழிமுறைகளை மாற்றித் தேர்வுசெய்யலாம்.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## முகவரியை இயக்குதல்

இப்போது நாமே முகவரியை இயக்கலாம். முகவரி உரையாடலை மடக்கங்களாக நினைவில் வைக்க `AgentSession` ஐ உருவாக்குகிறோம், அப்பிறகு இரண்டு `user_inputs` அனுப்புகிறோம். முதல் ஒன்று ஒரு பயணத்தை கேட்கிறது; இரண்டாவது பயனர் பரிந்துரையை விரும்பவில்லை என்று கூறுகிறது மற்றும் வேறொன்றை கேட்கிறது — முகவரி மடக்கு வரலாறுடன் கூடியது மற்றும் `get_random_destination` கருவியை பயன்படுத்தி பதிலளிக்கிறது.

நீங்கள் இந்த செய்திகளை மாற்றி முகவரி எப்படி வேறுபட்டதாக பதிலளிக்கிறது என்பதை கவனிக்கலாம். பதில்கள் **ஒவ்வொரு குறியீட்டாகவும்** ஊற்றப்படுகின்றன.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
